# Lecture 3 — Connecting to Databases

This notebook demonstrates how to connect to a SQLite database from Python, run SQL queries,
and load results into pandas DataFrames.

We use a small retail company sales database (`retail.db`) that contains the
following tables: Region, Store, Customer, Vendor, Category, Product, SalesTransaction, and Includes.
The database file is stored in the `data/` folder.

**Credits:**

Created by Sippo Rossi for the course Python Programming for Business Intelligence at Hanken.

## 1. Connecting to a SQLite database

Python's built-in `sqlite3` module lets us connect to SQLite databases without installing
any additional packages.

In [1]:
import sqlite3

In [40]:
# Connect to the database file
connection = sqlite3.connect("data/retail.db")
db = connection.cursor()

db.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = db.fetchall()

for table in tables:
    print(table[0])

vendor
category
product
region
store
customer
salestransaction
includes


The variable `db` now represents our connection to the database. We use it to send
SQL commands and retrieve results.

## 2. Running SQL queries with `execute()`

### 2.1 `fetchone()` - Retrieve a single result

`fetchone()` returns the first row of the query result as a tuple.

In [49]:
# Get the most expensive product
result = db.execute("SELECT MAX(ProductPrice) FROM Product").fetchone()
print(result)
print("Most expensive product costs:", result[0])

(500,)
Most expensive product costs: 500


### 2.2 `fetchall()` - Retrieve all results

`fetchall()` returns all rows as a list of tuples.

In [42]:
# Get the first 3 products with their prices
rows = db.execute("SELECT ProductName, ProductPrice FROM Product LIMIT 3").fetchall()
print(rows)

[('Zzz Bag', 100), ('Easy Boot', 70), ('Cosy Sock', 15)]


In [43]:
# We can loop through the results
for row in rows:
    print(f"{row[0]}: {row[1]}")

Zzz Bag: 100
Easy Boot: 70
Cosy Sock: 15


## 3. Using parameters in queries

Parameters allow us to feed values into queries safely. Use `?` as a placeholder
and pass the values as a list.

In [44]:
# Query a specific product by name
name = "Easy Boot"
price = db.execute(
    "SELECT ProductPrice FROM Product WHERE ProductName=?", [name]
).fetchone()

print(f"The price of {name} is {price[0]}")

The price of Easy Boot is 70


In [45]:
# Multiple parameters
min_price = 50
max_price = 200
products = db.execute(
    "SELECT ProductName, ProductPrice FROM Product WHERE ProductPrice BETWEEN ? AND ?",
    [min_price, max_price]
).fetchall()

for p in products:
    print(f"{p[0]}: {p[1]}")

Zzz Bag: 100
Easy Boot: 70
Dura Boot: 90
Tiny Tent: 150
Comfy Harness: 150
Sunny Charger: 125
Mmm Stove: 80
Bucky Knife: 60
Simple Sandal: 50
Action Sandal: 70


## 4. Modifying the database

We can also insert, update, and delete records. Note that changes are not saved
to the file until we call `db.commit()`.

In [60]:
# Insert a new vendor
db.execute("INSERT INTO Vendor (VendorID, VendorName) VALUES ('NI','Nike')")

# Check that it was added
vendors = db.execute("SELECT * FROM Vendor").fetchall()
for v in vendors:
    print(v)

('PG', 'Pacifica Gear')
('MK', 'Mountain King')
('OA', 'Outdoor Adventures')
('WL', 'Wilderness Limited')
('NI', 'Nike')


In [62]:
# Roll back the change so we don't permanently modify the demo database
connection.rollback()

# Checking that we return to the original state
vendors = db.execute("SELECT * FROM Vendor").fetchall()
for v in vendors:
    print(v)

('PG', 'Pacifica Gear')
('MK', 'Mountain King')
('OA', 'Outdoor Adventures')
('WL', 'Wilderness Limited')


## 5. Error handling with try and except

If a SQL command fails (e.g. trying to create a table that already exists), the program
would normally stop. We can use `try` and `except` to handle such errors gracefully.

In [63]:
try:
    db.execute("CREATE TABLE vendor (vendorid INTEGER PRIMARY KEY, vendorname TEXT)")
except:
    print("Cannot execute the given command (table likely already exists)")

Cannot execute the given command (table likely already exists)


## 6. Performance considerations

Databases are optimised for operations on large datasets. It is better to let the database
do as much work as possible rather than pulling raw data into Python and processing it there.

### 6.1 Let the database do the computation

Compare these two approaches for finding the most expensive product:

In [74]:
%%time

# Approach 1: Let SQL do the work (better)
most_expensive = db.execute("SELECT MAX(ProductPrice) FROM Product").fetchone()
print("SQL approach:", most_expensive[0])

SQL approach: 500
CPU times: user 293 μs, sys: 144 μs, total: 437 μs
Wall time: 320 μs


In [75]:
%%time

# Approach 2: Pull all prices into Python, then compute (worse)
prices = db.execute("SELECT ProductPrice FROM Product").fetchall()
most_expensive = max(prices)
print("Python approach:", most_expensive[0])

Python approach: 500
CPU times: user 495 μs, sys: 479 μs, total: 974 μs
Wall time: 657 μs


Both produce the correct answer, but Approach 1 is preferable because it avoids
transferring potentially millions of rows out of the database.

### 6.2 Minimise the number of queries

Avoid running queries inside a loop when a single query with a JOIN can achieve the same result.

In [78]:
%%time

# Bad: one query per vendor (N + 1 queries)
vendors = db.execute("SELECT vendorid FROM Vendor").fetchall()
for vendor in vendors:
    amount = db.execute(
        "SELECT COUNT(*) FROM Product WHERE vendorid=?", [vendor[0]]
    ).fetchone()
    print(vendor[0], amount[0])

MK 8
OA 5
PG 6
WL 5
CPU times: user 654 μs, sys: 924 μs, total: 1.58 ms
Wall time: 918 μs


In [79]:
%%time

# Good: a single query using a JOIN
data = db.execute(
    "SELECT v.vendorid, COUNT(p.ProductID) "
    "FROM vendor v LEFT JOIN product p ON v.vendorid = p.vendorid "
    "GROUP BY v.vendorid"
).fetchall()

for row in data:
    print(row[0], row[1])

MK 8
OA 5
PG 6
WL 5
CPU times: user 501 μs, sys: 439 μs, total: 940 μs
Wall time: 462 μs


The second approach can be several times faster, and the difference grows
as the number of vendors and products increases.

## 7. Loading query results into pandas

In [82]:
import pandas as pd

`pd.read_sql_query()` executes a SQL query and returns the result directly as a DataFrame.
This is often the most convenient way to bring database data into a pandas workflow.

In [85]:
connection = sqlite3.connect("data/retail.db")

df = pd.read_sql_query("SELECT * FROM Product", connection)
df.head()

,productid,productname,productprice,vendorid,categoryid
0,1X1,Zzz Bag,100,PG,CP
1,2X2,Easy Boot,70,MK,FW
2,3X3,Cosy Sock,15,MK,FW
3,4X4,Dura Boot,90,PG,FW
4,5X5,Tiny Tent,150,MK,CP


In [87]:
# A more targeted query
df_expensive = pd.read_sql_query(
    "SELECT ProductName, ProductPrice FROM Product WHERE ProductPrice > 50",
    connection
)
df_expensive

,productname,productprice
0,Zzz Bag,100
1,Easy Boot,70
2,Dura Boot,90
3,Tiny Tent,150
4,Biggy Tent,250
5,Hi-Tec GPS,300
6,Comfy Harness,150
7,Sunny Charger,125
8,Mmm Stove,80
9,Bucky Knife,60


### sqlite3 vs pandas

| Aspect | `sqlite3` | `pd.read_sql_query()` |
|---|---|---|
| Return type | List of tuples | DataFrame |
| Convenience | Lower-level, more control | Higher-level, easier to work with |
| Speed | Faster for raw queries | Slight overhead from DataFrame creation |

In practice, use `pd.read_sql_query()` when you plan to analyse the data with pandas,
and `sqlite3` directly when you need maximum control or performance.

## 8. Closing the connection

In [90]:
# Always close the database connection when done
db.close()
connection.close()
print("Connection closed")

Connection closed
